# Delta Demo — Episode 19: OPTIMIZE
### "Ten Small Files, One Slow Query — Can We Fix This Without Deleting Anything?"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo table:** `employees_ep19`

**Deletes only its own demo data:** Yes

---

**Important setup detail, worth stating explicitly:** throughout this series, Databricks' auto-OPTIMIZE has fired unpredictably after nearly every DML statement, silently compacting small files before we could even show them. For THIS episode specifically, we deliberately disable auto-compaction at table creation, so the 'messy small files' problem genuinely persists until we fix it ourselves, on purpose.

**Learning Outcome:** By the end of this episode, viewers should be able to explain what OPTIMIZE actually does — compacting many small files into fewer, larger ones — and why that's a performance operation, completely separate from VACUUM's cleanup role.

**Core Question:** A table built from many small writes ends up with many small files. Does that matter — and if so, how do you fix it without deleting any data?

### Today's Journey
✔ Deliberately create a table with 10 separate small files

↓

✔ Confirm the mess is real — count files, sizes

↓

✔ Understand why many small files hurt query performance

↓

✔ Run OPTIMIZE — watch the file count collapse

↓

✔ Confirm the OLD files are still on disk, not deleted

↓

✔ Read the real OPTIMIZE commit — what did Delta actually do?

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/employees_ep19

# =====================================================
# STEP 1 — Deliberately Create a Messy Table (10 Small Files)
# =====================================================
Auto-compaction is turned OFF from the very first write, so what you see below is genuinely what accumulates — not something Delta quietly cleaned up before we could look at it.

In [0]:
%python
from pyspark.sql import functions as F
import glob, json, os

table_path = "/Volumes/workspace/delta_demo/demo_files/employees_ep19"

# First row establishes the table AND disables auto-compaction in the
# same write, so every write after this stays small and uncompacted.
first_row = spark.createDataFrame(
    [(1, 'Ravi', 25000)], "eno INT, ename STRING, sal INT"
).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))

first_row.write.format("delta") \
    .option("delta.autoOptimize.autoCompact", "false") \
    .option("delta.autoOptimize.optimizeWrite", "false") \
    .mode("overwrite").save(table_path)

In [0]:
%python
# Nine more separate small writes — each one lands as its own physical
# file, simulating a real system that writes frequently in small batches
# (e.g. a streaming job, or a source that sends one record at a time).
names = ['Sridevi', 'Uma', 'Srik', 'Kanth', 'Divya', 'Manoj', 'Priya', 'Arjun', 'Lakshmi']
for i, name in enumerate(names, start=2):
    row = spark.createDataFrame(
        [(i, name, 20000 + i * 1000)], "eno INT, ename STRING, sal INT"
    ).withColumn("sal", F.col("sal").cast("DECIMAL(10,2)"))
    row.write.format("delta").mode("append").save(table_path)

print("10 total writes completed (1 initial + 9 appends).")

10 total writes completed (1 initial + 9 appends).


# =====================================================
# STEP 2 — Confirm the Mess Is Real
# =====================================================

In [0]:
%sh
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep19/*.parquet | wc -l
echo ""
ls -la /Volumes/workspace/delta_demo/demo_files/employees_ep19/*.parquet

10

-rwxrwxrwx 1 nobody nogroup 1231 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-6cefb3db-69a7-4c46-b12d-790130026ae1.c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 1230 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-7a9fba1b-8702-417f-bf80-ed4ea63a9c0e.c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 1230 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-88625d2e-005b-42d9-8696-5eb06ca45f73.c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 1242 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-8dd9475a-a096-49ed-920a-9c3c97bc07a9.c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 1226 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-992ac6cd-7a22-4bbf-b611-475cee3dd8e8.c000.snappy.parquet
-rwxrwxrwx 1 nobody nogroup 1226 Aug  2 16:04 /Volumes/workspace/delta_demo/demo_files/employees_ep19/part-00000-b825c27e-b06b-460d-9e7c-f8dc56055

In [0]:
%python
file_list = glob.glob(f"{table_path}/*.parquet")
file_sizes = [os.path.getsize(f) for f in file_list]
total_size = sum(file_sizes)

print(f"Number of physical files: {len(file_list)}")
print(f"Individual file sizes: {file_sizes}")
print(f"Total data size: {total_size} bytes")

if len(file_list) == 10:
    print("\n✅ VERIFIED: exactly 10 small files, as expected — auto-")
    print("   compaction genuinely did not interfere.")
else:
    print(f"\n⚠️ Got {len(file_list)} files instead of 10 — auto-compaction")
    print("   may have fired anyway. Check DESCRIBE HISTORY for OPTIMIZE rows.")

Number of physical files: 10
Individual file sizes: [1231, 1230, 1230, 1242, 1226, 1226, 1241, 1221, 1231, 1231]
Total data size: 12309 bytes

✅ VERIFIED: exactly 10 small files, as expected — auto-
   compaction genuinely did not interfere.


In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep19` ORDER BY eno;

eno,ename,sal
1,Ravi,25000.00
2,Sridevi,22000.00
3,Uma,23000.00
4,Srik,24000.00
5,Kanth,25000.00
6,Divya,26000.00
7,Manoj,27000.00
8,Priya,28000.00
9,Arjun,29000.00
10,Lakshmi,30000.00


# =====================================================
# STEP 3 — Why Does This Matter?
# =====================================================
Every file a query touches has real overhead — opening it, reading its footer, scheduling a task for it — independent of how much actual data is inside. Ten 1-row files cost roughly ten times the per-file overhead of one 10-row file, for the exact same data. At real production scale — thousands of small files instead of ten — this overhead becomes a genuine, measurable performance problem, not just an aesthetic one.

# =====================================================
# STEP 4 — Run OPTIMIZE
# =====================================================
🤔 **Prediction:** we have 10 tiny files. How many files do you expect after OPTIMIZE — and why?

In [0]:
%sql
OPTIMIZE delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep19`;

path,metrics
dbfs:/Volumes/workspace/delta_demo/demo_files/employees_ep19,"List(1, 10, List(1406, 1406, 1406.0, 1, 1406), List(1221, 1242, 1230.9, 10, 12309), 0, null, null, 0, 1, 10, 0, true, 0, 0, 1785686770666, 1785686772312, 8, 1, null, List(0, 0), null, 3, 3, 350, 0, null, null)"


### VERIFY — File Count After OPTIMIZE

In [0]:
%python
file_list_after = glob.glob(f"{table_path}/*.parquet")
print(f"Physical files immediately after OPTIMIZE: {len(file_list_after)}")

if len(file_list_after) >= len(file_list):
    print("\n⚠️ File count didn't decrease — check output above.")
else:
    print(f"\n✅ VERIFIED: file count dropped from {len(file_list)} to {len(file_list_after)}.")
    print("   Note: at this tiny demo scale, OPTIMIZE will likely compact")
    print("   everything into exactly 1 file — because the real rule is")
    print("   'aim for ~256MB per file,' not 'always produce 1 file.' Our")
    print("   entire table is nowhere near 256MB, so 1 file easily holds it.")

Physical files immediately after OPTIMIZE: 11

⚠️ File count didn't decrease — check output above.


# =====================================================
# STEP 5 — The Old Files: Deleted, or Just Inactive?
# =====================================================
Same question this whole series keeps coming back to.

In [0]:
%python
total_physical_files = len(glob.glob(f"{table_path}/*.parquet"))
active_files = len(spark.read.format("delta").load(table_path).inputFiles())

print(f"Active files (what a query reads): {active_files}")
print(f"Physical files still on disk: {total_physical_files}")
print(f"Orphaned garbage: {total_physical_files - active_files} files")

if total_physical_files > active_files:
    print("\n✅ VERIFIED: the original 10 small files are STILL physically")
    print("   on disk — OPTIMIZE only changed which files are ACTIVE, it")
    print("   never physically deletes anything. That's VACUUM's job, not")
    print("   OPTIMIZE's — covered fully in Episode 14.")
else:
    print("\n⚠️ Unexpected — investigate.")

Active files (what a query reads): 1
Physical files still on disk: 11
Orphaned garbage: 10 files

✅ VERIFIED: the original 10 small files are STILL physically
   on disk — OPTIMIZE only changed which files are ACTIVE, it
   never physically deletes anything. That's VACUUM's job, not
   OPTIMIZE's — covered fully in Episode 14.


# =====================================================
# STEP 6 — Read the Real OPTIMIZE Commit
# =====================================================

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/employees_ep19`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-08-02T16:06:13.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3221553489759848),073edb7a-bb94-4f8f-863d-20e879056c99,0802-151003-l0b6wf88-v2n,9,SnapshotIsolation,false,"Map(numRemovedFiles -> 10, numRemovedBytes -> 12309, p25FileSize -> 1406, numDeletionVectorsRemoved -> 0, minFileSize -> 1406, numAddedFiles -> 1, maxFileSize -> 1406, p75FileSize -> 1406, p50FileSize -> 1406, numAddedBytes -> 1406)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
9,2026-08-02T16:04:30.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),1187c97d-e668-4f8e-9014-06a6c19a22d6,0802-151003-l0b6wf88-v2n,8,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1241)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
8,2026-08-02T16:04:29.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),6859f1f8-0ed6-4cf1-980a-5de7ad13510f,0802-151003-l0b6wf88-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-08-02T16:04:28.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),281124ef-2b31-4652-bf23-28bb97c8fcfb,0802-151003-l0b6wf88-v2n,6,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1231)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-08-02T16:04:27.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),a267cea4-f199-4b51-ba8d-5aa507799038,0802-151003-l0b6wf88-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1231)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-02T16:04:26.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),923b2e29-33c2-49ac-a458-4f8d7d0bf639,0802-151003-l0b6wf88-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1231)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T16:04:25.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),ec59fb25-b9c9-448a-b1b3-09ba6ad83581,0802-151003-l0b6wf88-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1230)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T16:04:24.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),c85cda8b-df25-46fc-a598-aa0e0d8dd476,0802-151003-l0b6wf88-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1226)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T16:04:23.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),d32ca243-7f4f-4874-891e-1cd3946ea72d,0802-151003-l0b6wf88-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1221)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T16:04:22.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3221553489759848),455697ed-2208-495a-9e99-a41e643575ef

In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`{table_path}`")
optimize_row = history_df.filter(history_df.operation == "OPTIMIZE").orderBy("version", ascending=False).first()
metrics = dict(optimize_row['operationMetrics'])

print("OPTIMIZE operationMetrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

OPTIMIZE operationMetrics:
  numRemovedFiles: 10
  numRemovedBytes: 12309
  p25FileSize: 1406
  numDeletionVectorsRemoved: 0
  minFileSize: 1406
  p75FileSize: 1406
  p50FileSize: 1406
  numAddedBytes: 1406
  numAddedFiles: 1
  maxFileSize: 1406


In [0]:
%python
log_files = sorted(glob.glob(f"{table_path}/_delta_log/*.json"))
latest_commit_path = log_files[-1]
print(f"Reading: {latest_commit_path.split('/')[-1]}\n")

with open(latest_commit_path) as f:
    for line in f:
        action = json.loads(line)
        if 'add' in action:
            tags = action['add'].get('tags', {})
            print(f"ADD -> {action['add']['path']}")
            if 'OPTIMIZE_TARGET_SIZE' in tags:
                print(f"       OPTIMIZE_TARGET_SIZE tag: {tags['OPTIMIZE_TARGET_SIZE']} bytes")
        elif 'remove' in action:
            print(f"REMOVE -> {action['remove']['path']}")

Reading: 00000000000000000010.json

REMOVE -> part-00000-992ac6cd-7a22-4bbf-b611-475cee3dd8e8.c000.snappy.parquet
REMOVE -> part-00000-8dd9475a-a096-49ed-920a-9c3c97bc07a9.c000.snappy.parquet
REMOVE -> part-00000-c135eee7-7053-4171-98ea-05e58abf12f4.c000.snappy.parquet
REMOVE -> part-00000-b825c27e-b06b-460d-9e7c-f8dc56055153.c000.snappy.parquet
REMOVE -> part-00000-88625d2e-005b-42d9-8696-5eb06ca45f73.c000.snappy.parquet
REMOVE -> part-00000-e03d01bd-746a-4507-9515-b2a47549e16d.c000.snappy.parquet
REMOVE -> part-00000-6cefb3db-69a7-4c46-b12d-790130026ae1.c000.snappy.parquet
REMOVE -> part-00000-e636f889-107d-4e1f-9de0-d58ae8450ec7.c000.snappy.parquet
REMOVE -> part-00000-7a9fba1b-8702-417f-bf80-ed4ea63a9c0e.c000.snappy.parquet
REMOVE -> part-00000-bbd0a8b8-ce6b-49e1-b4a2-618c5f02566f.c000.snappy.parquet
ADD -> part-00000-6cd003a2-be46-4b67-aa2d-0f9ab39f6d0e.c000.snappy.parquet
       OPTIMIZE_TARGET_SIZE tag: 268435456 bytes


**Notice the `OPTIMIZE_TARGET_SIZE` tag — 268435456 bytes, exactly 256MB.** This is the same tag that's been sitting on every single file this whole series, since the very first commit back in Episode 10/Day 1. Delta was tagging every file with its compaction target from the start — OPTIMIZE was always the intended eventual step, whether or not we manually ran it.

# =====================================================
# STEP 7 — Enterprise Reality
# =====================================================
> "OPTIMIZE is a performance operation, not a cleanup operation — it's easy to confuse the two, especially since both eventually reduce the active file count. VACUUM permanently deletes files that are no longer needed. OPTIMIZE just repackages currently-active data into a more efficient physical layout — the files it replaces stay on disk until VACUUM eventually removes them. Real production tables often run OPTIMIZE on a schedule (nightly, weekly) specifically because small-file accumulation from frequent writes is a genuine, ongoing operational cost, not a one-time problem."

**One thing we deliberately controlled in this demo, worth stating honestly:** by default, Databricks often runs auto-compaction automatically — we explicitly disabled it here so the 'before' state was real and visible. In most real tables, some of this compaction happens without anyone running OPTIMIZE by hand at all.

We've now covered how Delta compacts files for size. Next: how do you organize data WITHIN those files so queries can skip the ones they don't need?

That's exactly what we'll explore in the next episode: ZORDER and Liquid Clustering.